In [1]:
import pandas as pd

In [179]:
def proxy_literacy_digital(df):
    total_penduduk = 1523065
    df = df.iloc[3:].reset_index(drop=True)

    header_row1 = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True).copy()
    df.columns = header_row1.values
    df.columns = df.columns.str.replace("'", "", regex=False)

    # df = df[df['KECAMATAN/KELURAHAN'] == 'KECAMATAN']

    #SUM UPPER SMA
    kolom_p = df.columns[df.columns.str.endswith('_P')]
    under_sma = ['TIDAK/BELUM SEKOLAH_P', 'BELUM TAMAT SD/SEDERAJAT_P', 'TAMAT SD/SEDERAJAT_P', 'SLTP/SEDERAJAT_P']
    upper_sma = [col for col in kolom_p if col not in under_sma]

    # Ambil hanya kolom WILAYAH dan hasil jumlah under SMA
    df = df[['WILAYAH'] + upper_sma]
    df['TOTAL'] = df[upper_sma].sum(axis=1)
    df['proxy_tingkat_literasi_digital'] = (df['TOTAL']/total_penduduk)*100

    return df



In [207]:
#LOAD DATA
df1 = pd.read_csv(r"D:\KULIAH\Project\womanguard-index-surabaya\DATA BARU\X7 (Tingkat Literasi Digital Perempuan)\2021 V\Banyaknya Penduduk Menurut Pendidikan dan Jenis Kelamin per Kecamatan Hasil Registrasi2021.csv", header = None)

In [208]:
# Ambil baris jenis pendidikan dan isi NaN ke kanan
pendidikan = df1.iloc[2].ffill()

In [209]:
# Ambil baris jenis kelamin: L, P, Total
gender = df1.iloc[4]

# Buat nama kolom baru
new_cols = ['WILAYAH']

In [210]:
for i in range(1, df1.shape[1]):
    col_name = f"{pendidikan[i]}_{gender[i]}"
    col_name = (
        col_name
        .replace("'", "")
        .replace("SLTA SEDERAJAT", "SLTA/SEDERAJAT")
        .strip()
        .upper()
    )
    new_cols.append(col_name)

In [211]:
# Ambil data mulai dari baris kecamatan
df_clean = df1.iloc[5:].copy()
df_clean.columns = new_cols

In [212]:
# Bersihkan nama wilayah
df_clean['WILAYAH'] = df_clean['WILAYAH'].astype(str).str.strip()

In [213]:
df_clean = df_clean[df_clean['WILAYAH'].str.upper() != 'KOTA SURABAYA']

In [214]:
# Ubah kolom angka menjadi numeric
num_cols = df_clean.columns.drop('WILAYAH')

df_clean[num_cols] = (
    df_clean[num_cols]
    .replace(r'[^\d.-]', '', regex=True)
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(int)
)

df_clean = df_clean.reset_index(drop=True)

In [194]:
df_clean

,WILAYAH,TIDAK/BELUM SEKOLAH_L,TIDAK/BELUM SEKOLAH_P,TIDAK/BELUM SEKOLAH_TOTAL,BELUM TAMAT SD/SEDERAJAT_L,BELUM TAMAT SD/SEDERAJAT_P,BELUM TAMAT SD/SEDERAJAT_TOTAL,TAMAT SD/SEDERAJAT_L,TAMAT SD/SEDERAJAT_P,TAMAT SD/SEDERAJAT_TOTAL,...,D3/SARJANA MUDA_TOTAL,D4/S1_L,D4/S1_P,D4/S1_TOTAL,S2_L,S2_P,S2_TOTAL,S3_L,S3_P,S3_TOTAL
0,Karangpilang,9073,8579,17652,3141,3164,6305,3013,4155,7168,...,1362,4357,4473,8830,352,272,624,21,15,36
1,Jambangan,6078,5746,11824,2421,2333,4754,1585,2414,3999,...,1318,4131,4244,8375,443,315,758,27,17,44
2,Gayungan,5708,5321,11029,1533,1525,3058,1341,1905,3246,...,945,3949,4016,7965,503,336,839,32,19,51
3,Wonocolo,9821,9354,19175,3425,3286,6711,2946,4212,7158,...,1490,5419,5488,10907,528,391,919,50,22,72
4,Tenggilis Mejoyo,6470,6158,12628,2673,2505,5178,2297,3212,5509,...,1170,4258,4412,8670,408,322,730,30,15,45
5,Gunung Anyar,7197,6701,13898,2737,2592,5329,2063,2851,4914,...,1331,5434,5470,10904,524,378,902,46,22,68
6,Rungkut,13735,12988,26723,4627,4429,9056,4425,5875,10300,...,2472,10085,10432,20517,1054,781,1835,121,69,190
7,Sukolilo,15875,14879,30754,3396,3340,6736,5966,7299,13265,...,1744,8783,9154,17937,1088,912,2000,210,87,297
8,Mulyorejo,11914,11068,22982,2582,2398,4980,3799,4939,8738,...,1312,7511,7691,15202,686,489,1175,60,47,107
9,Gubeng,15111,14086,29197,5931,5702,11633,4463,6824,11287,...,2975,9647,10091,19738,730,531,1261,61,33,94


In [215]:
total_penduduk = 1498135
under_sma = [
    'TIDAK/BELUM SEKOLAH_P',
    'BELUM TAMAT SD/SEDERAJAT_P',
    'TAMAT SD/SEDERAJAT_P',
    'SLTP/SEDERAJAT_P',
]
upper_sma = [
    col for col in df_clean.columns
    if col.endswith('_P') and col not in under_sma
]

data = df_clean[['WILAYAH'] + upper_sma].copy()
data['TOTAL'] = data[upper_sma].sum(axis=1)
data['proxy_tingkat_literasi_digital'] = (data['TOTAL'] / total_penduduk) * 100

In [217]:
data.to_csv(r"D:\KULIAH\Project\womanguard-index-surabaya\DATA BARU\X7 (Tingkat Literasi Digital Perempuan)\2021 V\tingkat_literasi_digital_2021.csv")

In [112]:
#DROP 4 kolom pertama
df1 = df1.iloc[3:].reset_index(drop=True)
df2 = df2.iloc[3:].reset_index(drop=True)

In [113]:
#Mengubah baris pertama jadi header
header_row1 = df1.iloc[0]
df1 = df1.iloc[1:].reset_index(drop=True).copy()
df1.columns = header_row1.values
df1.columns = df1.columns.str.replace("'", "", regex=False)

header_row2 = df2.iloc[0]
df2 = df2.iloc[1:].reset_index(drop=True).copy()
df2.columns = header_row2.values
df2.columns = df2.columns.str.replace("'", "", regex=False)

In [114]:
#Ambil kecamatan aja
df1 = df1[df1['KECAMATAN/KELURAHAN'] == 'KECAMATAN']
df2 = df2[df2['KECAMATAN/KELURAHAN'] == 'KECAMATAN']

In [116]:
#SUM UNDER SMA
kolom_p = df1.columns[df1.columns.str.endswith('_P')]
under_sma = ['TIDAK/BELUM SEKOLAH_P', 'BELUM TAMAT SD/SEDERAJAT_P', 'TAMAT SD/SEDERAJAT_P', 'SLTP/SEDERAJAT_P']
upper_sma = [col for col in kolom_p if col not in under_sma]
# Gabungkan berdasarkan kecamatan/kelurahan
df_concat = df1.merge(
    df2,
    on='WILAYAH',
    suffixes=('_df1', '_df2')
)

# Jumlahkan masing-masing kolom
for col in upper_sma:
    df_concat[col] = df_concat[f'{col}_df1'] + df_concat[f'{col}_df2']

# Ambil hanya kolom WILAYAH dan hasil jumlah under SMA
df_concat = df_concat[['WILAYAH'] + upper_sma]

In [117]:
df_concat['TOTAL'] = df_concat[upper_sma].sum(axis=1)

In [120]:
df_concat['proxy_tingkat_literasi_digital'] = (df_concat['TOTAL']/1516783)*100

In [123]:
df_concat.to_csv(r"D:\KULIAH\Project\womanguard-index-surabaya\DATA BARU\X7 (Tingkat Literasi Digital Perempuan)\2025\tingkat_literasi_digital_2025.csv")